# Task 1.3 — What the Paper Claims to Improve

**Selected Paper:** Learning Kernels with Radiuses of Minimum Enclosing Balls (NeurIPS 2010)

This notebook compares the baseline Margin-based Multiple Kernel Learning (MKL) with the proposed Radius-based Kernel Learning (RKL), explains the limitations the paper identifies, and discusses a scenario where RKL may not outperform the baseline.

---
## 1. Baseline Method — Margin-Based Multiple Kernel Learning

The baseline is **margin-based Multiple Kernel Learning (MKL)**, which learns a kernel by maximising the SVM margin. Given $M$ basis kernels $k_1, \ldots, k_M$, the learned kernel is a nonnegative linear combination $k^{(\theta)} = \sum_j \theta_j k_j$ (Eq. 2). The MKL formulation jointly selects the kernel weights $\theta$ and trains an SVM classifier by solving (Section 2, Eq. 3):

$$\min_{\theta \in \Theta} \; \min_{w,\, b,\, \xi} \; \frac{1}{2} \|w\|^2 + C \sum_i \xi_i$$
$$\text{s.t.} \quad y_i(\langle \phi(x_i;\, k^{(\theta)}),\, w \rangle + b) + \xi_i \geq 1, \quad \xi_i \geq 0$$

where $\Theta$ constrains $\theta$ (e.g., $\theta_j \geq 0$, $\|\theta\|_1 = 1$ or $\|\theta\|_2 = 1$). The core idea behind this objective is the **large margin principle**: minimising $\|w\|^2$ is equivalent to maximising the SVM margin $\gamma = 1/\|w\|$, so the formulation searches for the kernel combination that yields the largest possible margin while keeping the empirical loss (controlled by the slack variables $\xi_i$ and the regularisation parameter $C$) small.

The paper evaluates against three concrete instantiations of this margin-based baseline:
- **Unif** — SVM with uniform weights $\theta_j = 1/M$, i.e., no kernel learning at all (Index 1, Table 1).
- **MKL** — Standard MKL that optimises $\theta$ under an L1 or L2 norm constraint (Indices 2 and 5, Table 1).
- **KL-C** — The formulation of Chapelle et al. [1], which attempts to incorporate the MEB radius by modifying the kernel matrix as $K(\theta) + \frac{1}{C}I$, but still ultimately relies on the margin and therefore remains sensitive to kernel scaling (Eq. 10; Indices 3 and 6, Table 1).

---
## 2. Limitations Identified by the Paper (Section 2.1)

The paper demonstrates that the margin-only objective $\widetilde{G}(k) = \frac{1}{2}\|w\|^2 + C\sum_i \xi_i$ suffers from two fundamental problems, both stemming from the fact that **the margin changes when the kernel is scaled, even though scaling does not change the underlying geometry of the data**.

### 2a. The Scaling Problem

Consider what happens when we multiply a kernel $k$ by a scalar $a > 1$ to get $k_{\text{new}} = ak$. The feature map scales as $\phi(\cdot\,; k_{\text{new}}) = \sqrt{a}\,\phi(\cdot\,; k)$. If we then set $w_2 = w^*/\sqrt{a}$, the decision boundary stays exactly the same (the classifier makes identical predictions), and the slack values $\xi_i$ are unchanged. But the regularisation term shrinks:

$$\|w_2\|^2 = \frac{\|w^*\|^2}{a} < \|w^*\|^2 \quad \Longrightarrow \quad \widetilde{G}(ak) < \widetilde{G}(k)$$

This means we can always make the objective smaller — and the margin larger — **just by multiplying the kernel by a bigger number**. Any kernel, no matter how poor it is for the actual classification task, can be made to look good simply by inflating its scale. The margin does not measure kernel quality; it measures kernel magnitude (Section 2.1).

### 2b. The Initialisation Problem

One natural fix is to add a norm constraint on $\theta$ (e.g., $\sum_j \theta_j = 1$) so that the weights cannot grow without bound. However, the paper shows this does not fully resolve the issue because the solution becomes **sensitive to how the basis kernels are initially scaled**.

The paper illustrates this with a clear example (Section 2.1, Eq. 6): suppose we have two basis kernels $k_1$ and $k_2$, where $k_1$ naturally gives a larger margin. Under the L1 constraint $\theta_1 + \theta_2 = 1$, MKL selects $k_1$. Now suppose someone simply rescales $k_1$ by a small factor, replacing it with $k_1' = a \cdot k_1$ where $a$ is small enough that the rescaled $k_1'$ has a smaller margin than $k_2$. MKL will now select $k_2$ instead — even though $k_1$ and $k_1'$ represent the same kernel up to a scale factor and induce the same geometry in feature space.

**Root cause of both problems:** The margin $\gamma = 1/\|w\|$ is not invariant to kernel scaling. This means that when we optimise the margin-based objective, we are partly optimising the scale of the kernel rather than its intrinsic quality for the classification task.

---
## 3. How RKL Solves These Problems (Section 3)

The proposed **Radius-based Kernel Learning (RKL)** formulation directly addresses both problems by replacing the margin-only regulariser $\|w\|^2$ with the product $R^2(k)\|w\|^2$, where $R^2(k)$ is the squared radius of the minimum enclosing ball (MEB) of the training data in the feature space of kernel $k$ (Eq. 8). The new objective is (Eq. 9):

$$\min_{k,\, w,\, b,\, \xi} \; \frac{1}{2} R^2(k) \|w\|^2 + C \sum_i \xi_i$$
$$\text{s.t.} \quad y_i(\langle \phi(x_i; k),\, w \rangle + b) + \xi_i \geq 1, \quad \xi_i \geq 0$$

The key insight is that $R^2(k)\|w\|^2$ is proportional to $R^2 / \gamma^2$ (where $\gamma = 1/\|w\|$ is the margin), which appears in SVM generalisation bounds and is a more reliable measure of kernel quality than the margin alone.

### Why this makes the optimisation invariant to kernel scaling (Proposition 1)

The MEB radius has the property $R^2(ak) = a \cdot R^2(k)$ — scaling the kernel by $a$ scales the radius by $a$. At the same time, the optimal $\|w\|^2$ scales by $1/a$ (since $w$ adjusts to maintain the same decision boundary). These two effects cancel exactly:

$$R^2(ak) \cdot \|w^*\|^2 = a \cdot R^2(k) \cdot \frac{\|w^*\|^2}{a} = R^2(k) \cdot \|w^*\|^2$$

so that $G(ak) = G(k)$ for all $a > 0$. **Scaling a kernel no longer changes the objective value**, which completely eliminates the scaling problem.

### Why this also fixes the initialisation problem (Propositions 2 & 3)

- **Proposition 2:** Because the objective is scaling-invariant, optimising $G$ with an L1 norm constraint, an L2 norm constraint, or no norm constraint at all on $\theta$ all produce **equivalent solutions**. The choice of norm constraint becomes irrelevant.
- **Proposition 3:** If individual basis kernels are rescaled by arbitrary positive factors $a_j$ (e.g., $k_j \to a_j k_j$), the optimal objective value and the set of local/global optima remain exactly the same. Different initial scalings of the basis kernels do not bias the solution.

Together, these invariance properties mean that RKL is **completely immune** to both the scaling and initialisation problems that undermine margin-based MKL.

---
## 4. A Scenario Where RKL May Not Outperform the Baseline

**Scenario: Outliers in feature space that inflate the MEB radius.**

The MEB radius $R^2(k)$ is defined as the squared radius of the smallest ball enclosing *all* training points in feature space (Eq. 7–8). This means that even a single outlier — a point that maps far from the rest of the data — can dominate the radius, making $R^2(k)$ a poor representation of the actual spread of the bulk of the data. When this happens, the RKL objective is distorted in two ways:

1. **Kernel selection is misguided:** The regulariser $R^2(k)\|w\|^2$ becomes driven by the outlier's position rather than the geometry of the main data clusters. The optimisation may then favour kernels that **shrink the overall feature space** (to reduce $R^2$) at the expense of maintaining good class separability.
2. **The baseline is more robust in this case:** The standard margin-based MKL objective $\|w\|^2 + C\sum_i \xi_i$ does not use the radius at all. Outliers affect it only through the slack variables $\xi_i$, which handle each misclassified or margin-violating point individually without inflating a global term in the objective.

Evidence from the paper supports this: on the **Wpbc** dataset (Table 1), RKL (Indices 4, 7, 8) achieves 76.5% accuracy — no better than Unif (76.5%, Index 1) or MKL (76.5%, Index 2). This is consistent with a dataset where the MEB radius does not carry useful information for distinguishing between kernels, so the additional complexity of RKL provides no benefit.

More broadly, whenever the generalisation bound $R^2/\gamma^2$ is loose — for example, with very small sample sizes, very high-dimensional feature spaces, or heavy-tailed data distributions — the radius-margin ratio may not correlate well with test error, and the simpler margin-based baseline can match or even exceed RKL's performance.